In [0]:
from pyspark.sql.functions import current_timestamp, lit

SOURCE_TABLE = "banking.banking.banking_raw_transactions"
TARGET_TABLE = "banking.banking.banking_bronze_transactions"

print("🔹 Starting Bronze Layer")

In [0]:
# Step 1: Read raw data
raw_df = spark.table(SOURCE_TABLE)

print(f"Raw count: {raw_df.count()}")


In [0]:
# Step 2: Check if Bronze table exists
table_exists = spark.catalog.tableExists(TARGET_TABLE)

if table_exists:
    print("Applying idempotency check")

    bronze_existing = spark.table(TARGET_TABLE)

    # Remove already processed records
    raw_df = raw_df.join(
        bronze_existing.select("txn_id"),
        on="txn_id",
        how="left_anti"
    )
else:
    print("First run - Bronze table will be created")

print(f"Records to process: {raw_df.count()}")

In [0]:
# Step 3: Add metadata
bronze_df = (
    raw_df
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_system", lit("banking_simulator"))
    .withColumn("record_status", lit("RAW"))
)


In [0]:
# Step 4: Write to Bronze
bronze_df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(TARGET_TABLE)

print("✅ Bronze load completed")


In [0]:
display(bronze_df)